In [1]:
import pandas as pd

out = './mbdump_small/'

real = pd.read_csv(f'{out}listening_history_real.tsv', sep='\t')
rec  = pd.read_csv(f'{out}recording', sep='\t', header=None,
       names=['id','gid','name','artist_credit','length','comment',
              'edits_pending','last_updated','video'])

print(f"real listens         → {len(real):,}")
print(f"real with mbid       → {real['recording_mbid'].notna().sum():,}")
print(f"recording in catalog → {len(rec):,}")

mbids_in_real    = set(real['recording_mbid'].dropna())
mbids_in_catalog = set(rec['gid'].dropna())
overlap = mbids_in_real & mbids_in_catalog
print(f"mbid overlap         → {len(overlap):,}")

# sample both to compare format
print("\nSample real mbids:")
print(list(mbids_in_real)[:3])
print("\nSample catalog gids:")
print(list(mbids_in_catalog)[:3])

real listens         → 3,743,372
real with mbid       → 19,321
recording in catalog → 79,359
mbid overlap         → 13,215

Sample real mbids:
['f22934ee-602c-4ac8-8683-86e5bdbf3aef', 'ddfbcc39-40fc-4a3e-ad0a-b07c33ee9bca', '4f610c37-8945-4a60-9421-b73f386ccaab']

Sample catalog gids:
['3e7b3167-f09c-4e75-81b2-c4ff071258ff', '2c4ec693-2c40-4039-9403-09f50bd37dbd', '2331d10e-832c-4c80-bd6f-79f2a887a2b0']


In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

out  = './mbdump_small/'
OPTS = dict(sep='\t', header=None, engine='python', on_bad_lines='skip', quoting=3)

random.seed(42)
np.random.seed(42)

# ── load catalog ───────────────────────────────────────────────────────────
recording = pd.read_csv(f'{out}recording',
    names=['id','gid','name','artist_credit','length','comment',
           'edits_pending','last_updated','video'], **OPTS)

artist = pd.read_csv(f'{out}artist',
    names=['id','gid','name','sort_name','begin_date_year','begin_date_month',
           'begin_date_day','end_date_year','end_date_month','end_date_day',
           'type','area','gender','comment','edits_pending','last_updated',
           'ended','begin_area','end_area'], **OPTS)

artist_credit_name = pd.read_csv(f'{out}artist_credit_name',
    names=['artist_credit','position','artist','name','join_phrase'], **OPTS)

release = pd.read_csv(f'{out}release',
    names=['id','gid','name','artist_credit','release_group','status',
           'packaging','language','script','barcode','comment',
           'edits_pending','quality','last_updated'], **OPTS)

artist_ids_list = list(artist['id'])

# ══════════════════════════════════════════════════════════════════════════
# 1. REAL LISTENING HISTORY — from ListenBrainz
# ══════════════════════════════════════════════════════════════════════════
print("Loading real listens...")
real = pd.read_csv(f'{out}listening_history_real.tsv', sep='\t')

# join recording_mbid → MusicBrainz recording.id
real = real.merge(
    recording[['id','gid']],
    left_on='recording_mbid', right_on='gid',
    how='inner'
).rename(columns={'id': 'recording_id'})

print(f"Matched {len(real):,} listens with MusicBrainz catalog")

# map real LB user_ids → contiguous synthetic user_ids 1..N
lb_users   = real['user_id'].unique()
N_USERS    = len(lb_users)
uid_map    = {lb: i+1 for i, lb in enumerate(lb_users)}
real['user_id'] = real['user_id'].map(uid_map)

# parse timestamp → time_of_day
real['timestamp'] = pd.to_datetime(real['timestamp'], unit='s', errors='coerce')
real = real.dropna(subset=['timestamp'])
real['time_of_day'] = real['timestamp'].dt.hour.map(
    lambda h: 'morning' if h < 12 else 'afternoon' if h < 18 else 'night'
)

# fill duration if missing
length_map = recording.set_index('id')['length']
real['duration_ms'] = real['duration_ms'].fillna(
    real['recording_id'].map(length_map)
)
real['completed'] = real['duration_ms'] > 120_000  # >2min = completed

history = real[['user_id','recording_id','timestamp','time_of_day','duration_ms','completed']].copy()
history = history.sort_values('timestamp').reset_index(drop=True)
print(f"history → {len(history):,} rows | {history['user_id'].nunique()} users")

# ══════════════════════════════════════════════════════════════════════════
# 2. USERS — built from real LB users
# ══════════════════════════════════════════════════════════════════════════
if 'user_name' in real.columns:
    usernames = (real.drop_duplicates('user_id')
                     .sort_values('user_id')['user_name']
                     .reset_index(drop=True))
else:
    usernames = [f'user_{i}' for i in range(1, N_USERS+1)]

users = pd.DataFrame({
    'user_id': range(1, N_USERS+1),
    'username': usernames,
    'age':     np.random.randint(18, 45, N_USERS),
    'country': np.random.choice(['PT','BR','US','UK','ES'], N_USERS)
})

# ══════════════════════════════════════════════════════════════════════════
# 3. FRIENDS GRAPH — shared artist taste bias
#    users who like same artists → more likely friends
# ══════════════════════════════════════════════════════════════════════════

# power-law artist popularity: few artists get many listens
artist_plays = (history
    .merge(recording[['id','artist_credit']], left_on='recording_id', right_on='id', how='left')
    .groupby('artist_credit').size()
    .reset_index(name='plays')
    .sort_values('plays', ascending=False))

# top artists per user
user_top_artists = (history
    .merge(recording[['id','artist_credit']], left_on='recording_id', right_on='id', how='left')
    .groupby(['user_id','artist_credit']).size()
    .reset_index(name='plays')
    .sort_values(['user_id','plays'], ascending=[True,False])
    .groupby('user_id').head(5))

user_artist_map = user_top_artists.groupby('user_id')['artist_credit'].apply(set).to_dict()

friends = []
friend_set = set()
uid_list = list(range(1, N_USERS+1))

for uid in uid_list:
    n_friends = random.randint(3, 15)
    pool = [u for u in uid_list if u != uid]

    # score candidates by shared top artists
    shared = []
    my_artists = user_artist_map.get(uid, set())
    for cand in pool:
        overlap = len(my_artists & user_artist_map.get(cand, set()))
        shared.append((cand, overlap + 0.1))  # 0.1 floor so all have chance

    cands, weights = zip(*shared)
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()

    chosen = np.random.choice(cands, size=min(n_friends, len(cands)),
                               replace=False, p=weights)
    for fid in chosen:
        if (fid, uid) not in friend_set:
            friend_set.add((uid, fid))
            friends.append({'user_id': uid, 'friend_id': fid})

friends = pd.DataFrame(friends)

# ══════════════════════════════════════════════════════════════════════════
# 4. FAVOURITE ARTISTS — power-law biased
#    pick from globally popular artists, not uniform random
# ══════════════════════════════════════════════════════════════════════════

# zipf weights over artist_ids_list by global play rank
n_artists   = len(artist_ids_list)
zipf_ranks  = np.arange(1, n_artists+1, dtype=float)
zipf_weights = 1.0 / zipf_ranks
zipf_weights /= zipf_weights.sum()

# sort artist_ids by play count (most played first)
popular_artist_ids = (artist_plays
    .merge(artist_credit_name[['artist_credit','artist']].drop_duplicates(),
           on='artist_credit', how='left')
    .dropna(subset=['artist'])['artist']
    .astype(int).tolist())

# pad with random if fewer than n_artists
if len(popular_artist_ids) < n_artists:
    missing = [a for a in artist_ids_list if a not in set(popular_artist_ids)]
    popular_artist_ids += missing
popular_artist_ids = popular_artist_ids[:n_artists]

fav_artists = []
for uid in users['user_id']:
    n_fav = random.randint(2, 8)
    chosen = np.random.choice(popular_artist_ids,
                               size=min(n_fav, len(popular_artist_ids)),
                               replace=False, p=zipf_weights[:len(popular_artist_ids)] /
                                               zipf_weights[:len(popular_artist_ids)].sum())
    for aid in chosen:
        fav_artists.append({'user_id': uid, 'artist_id': int(aid)})

fav_artists = pd.DataFrame(fav_artists)

# ══════════════════════════════════════════════════════════════════════════
# 5. STREAMING EVENTS — from real history
# ══════════════════════════════════════════════════════════════════════════
events = []
for _, row in history.iterrows():
    events.append({
        'event_type':   'play',
        'user_id':      row['user_id'],
        'recording_id': row['recording_id'],
        'ts':           row['timestamp'].isoformat(),
        'duration_ms':  row['duration_ms']
    })
    if not row['completed']:
        events.append({
            'event_type':   'skip',
            'user_id':      row['user_id'],
            'recording_id': row['recording_id'],
            'ts':           row['timestamp'].isoformat(),
            'position_ms':  random.randint(5000, 60000)
        })
events_df = pd.DataFrame(events)

# ══════════════════════════════════════════════════════════════════════════
# 6. NOTIFICATIONS — real releases × fav artists
# ══════════════════════════════════════════════════════════════════════════
notifications = []
for _, row in fav_artists.iterrows():
    artist_releases = release[release['artist_credit'].isin(
        artist_credit_name[artist_credit_name['artist'] == row['artist_id']]['artist_credit']
    )]
    for _, rel in artist_releases.iterrows():
        notifications.append({
            'user_id':     row['user_id'],
            'artist_id':   row['artist_id'],
            'release_id':  rel['id'],
            'release_name': rel['name'],
            'notified_at': (datetime.now() - timedelta(days=random.randint(0,30))).isoformat()
        })
notifications = pd.DataFrame(notifications)

# ══════════════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════════════
users.to_csv(         f'{out}users.tsv',              sep='\t', index=False)
friends.to_csv(       f'{out}friends.tsv',            sep='\t', index=False)
fav_artists.to_csv(   f'{out}fav_artists.tsv',        sep='\t', index=False)
history.to_csv(       f'{out}listening_history.tsv',  sep='\t', index=False)
events_df.to_csv(     f'{out}streaming_events.tsv',   sep='\t', index=False)
notifications.to_csv( f'{out}notifications.tsv',      sep='\t', index=False)

with pd.ExcelWriter(f'{out}synthetic_data.xlsx') as writer:
    users.to_excel(        writer, sheet_name='users',         index=False)
    friends.to_excel(      writer, sheet_name='friends',       index=False)
    fav_artists.to_excel(  writer, sheet_name='fav_artists',   index=False)
    history.to_excel(      writer, sheet_name='history',       index=False)
    events_df.to_excel(    writer, sheet_name='streaming',     index=False)
    notifications.to_excel(writer, sheet_name='notifications', index=False)

for name, df in [('users',users),('friends',friends),('fav_artists',fav_artists),
                 ('history',history),('streaming_events',events_df),
                 ('notifications',notifications)]:
    print(f"{name:25s} → {len(df)} rows")

Loading real listens...
Matched 18,512 listens with MusicBrainz catalog
history → 18,512 rows | 175 users
users                     → 175 rows
friends                   → 1514 rows
fav_artists               → 858 rows
history                   → 18512 rows
streaming_events          → 21600 rows
notifications             → 115460 rows
